In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings('ignore')

DIRS = [
    "data_processed/split_definitions", 
    "results/models", "results/metrics", "results/predictions", 
    "results/tables", "manuscript_assets", "supplementary_assets"
]
for d in DIRS:
    Path(d).mkdir(parents=True, exist_ok=True)

    
DATA_FILE = "clean_data.csv"
ID_COL = "filename"
TARGET_COL = "uptake(mmol/g) methane at 65 bar"
PREDS_FILE = "results/predictions/Table5_random_preds.csv"
RAW_METRICS_FILE = "results/metrics/Table5_random_raw.csv"

# Ensure SI directories exist
Path("supplementary_assets").mkdir(parents=True, exist_ok=True)
Path("results/tables/robustness").mkdir(parents=True, exist_ok=True)

# Re-create the baseline features to match Script 1
GEOMETRY_COLS = ['Density', 'AVAf', 'AVA', 'ASA', 'Df', 'Di']
ENRICHED_COLS = GEOMETRY_COLS + [
    'UC_volume', 'vASA', 'NASA', 'POAVA', 
    'lcd_pld_ratio', 'cavity_window_gap', 'sa_pv_ratio', 'vf_density_ratio', 'log_pld_plus1', 'log_lcd_plus1'
]
m=3

params = {
    # Font family
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],

    # Font sizes
    "axes.labelsize": 10*m,
    "font.size": 10*m,
    "legend.fontsize": 10*m,
    "xtick.labelsize": 9*m,
    "ytick.labelsize": 9*m,

    # Style for axis labels (xlabel, ylabel)
    'axes.labelweight': 'bold',
    'axes.labelcolor': 'black',

    # General styles for other elements
    'font.weight': 'bold',       # Makes title, etc., bold
    'xtick.color': 'black',      # Sets tick label color
    'ytick.color': 'black',
    'legend.labelcolor': 'black'
}

plt.rcParams.update(params)

def optimize_data_types(df):
    # Convert float64 to float32 (halves memory usage)
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].astype('float32')
    
    # Convert int64 to int32 or int16
    int_cols = df.select_dtypes(include=['int64']).columns
    for col in int_cols:
        if df[col].max() < 32767 and df[col].min() > -32768:
            df[col] = df[col].astype('int16')
        elif df[col].max() < 2147483647 and df[col].min() > -2147483648:
            df[col] = df[col].astype('int32')
    
    # Convert object columns to categorical if they have few unique values
    obj_cols = df.select_dtypes(include=['object']).columns
    for col in obj_cols:
        if df[col].nunique() / len(df) < 0.5:  # If less than 50% unique
            df[col] = df[col].astype('category')
    
    return df

        
def load_and_prep_data():
    """Loads raw data and re-applies the exact same feature engineering."""
    df = pd.read_csv(DATA_FILE).dropna(subset=[TARGET_COL]).drop_duplicates(subset=[ID_COL])
    df = df[(df['Di'] >= 0) & (df['Df'] >= 0) & (df['Density'] >= 0) & (df['ASA'] >= 0)].copy()
    
    eps = 1e-8
    df['lcd_pld_ratio'] = df['Di'] / (df['Df'] + eps)
    df['cavity_window_gap'] = df['Dif']
    df['sa_pv_ratio'] = df['ASA'] / (df['AVA'] + eps)
    df['vf_density_ratio'] = df['AVAf'] / (df['Density'] + eps)
    df['log_pld_plus1'] = np.log1p(df['Df'].clip(lower=0))
    df['log_lcd_plus1'] = np.log1p(df['Di'].clip(lower=0))
    
    # Topology handling
    topo_counts = df['Crystalnet'].value_counts(dropna=False)
    top_topos = set(topo_counts.nlargest(20).index)
    df['topology_label'] = df['Crystalnet'].apply(lambda x: x if x in top_topos else 'other')
    
    return df

# ==========================================
# 1. SI FIGURES (S1 - S6)
# ==========================================
def generate_si_figures(df, preds, metrics):
    print("Generating SI Figures 1-6...")
    df = optimize_data_types(df)
    preds = optimize_data_types(preds)
    merged = preds.merge(df, on=ID_COL, how="left")
    merged['abs_error'] = (merged['y_true'] - merged['y_pred']).abs()
    merged['pld_decile'] = pd.qcut(merged['Df'], 10, labels=False, duplicates='drop')

    # -- SI Figure 1: Target Distribution --
    plt.figure(figsize=(8, 5))
    sns.histplot(df[TARGET_COL], kde=True, bins=50, color='teal')

    plt.xlabel(f"uptake(mmol/g)")
    plt.tight_layout()
    plt.savefig("supplementary_assets/SI_figure1_target_dist.pdf")
    plt.close()

    # -- SI Figure 2: Pairwise Correlation (Enriched Geometry) --
    plt.figure(figsize=(12, 10))
    corr = df[ENRICHED_COLS].corr()
    sns.heatmap(corr, annot=False, cmap='coolwarm', center=0, vmin=-1, vmax=1)

    plt.tight_layout()
    plt.savefig("supplementary_assets/SI_figure2_correlation.pdf")
    plt.close()

    # -- SI Figure 3: Seed Stability --
    plt.figure(figsize=(16*0.8, 9*0.7))
    ax = sns.boxplot(data=metrics, x='family', y='r2', hue='model')
    
    # MAPPING: Convert family names to desired labels
    label_mapping = {
        'geometry_only': 'Geo',
        'enriched_interpretable': 'Enriched',
        'topology_only': 'Topo',
        'geometry_plus_topology': 'Geo + Topo'
    }
    
    # Set x-axis tick labels in EXACT desired order
    ax.set_xticklabels([label_mapping[x] for x in ['geometry_only', 'enriched_interpretable', 'topology_only', 'geometry_plus_topology']])
    
    plt.ylabel("$R^2$ ")
    plt.legend(bbox_to_anchor=(1.05, 0.5), loc='center left')
    plt.tight_layout()
    plt.savefig("supplementary_assets/SI_figure3_seed_stability.pdf")
    plt.close()

    # -- SI Figure 4: Regime Performance (PLD Lineplot) --
    hgb_data = merged[merged['model'] == 'hgb']
    plt.figure(figsize=(10, 8))
    ax = sns.lineplot(
        data=hgb_data, 
        x='pld_decile', 
        y='abs_error', 
        hue='family', 
        marker=None, 
        errorbar=None
    )

# Create custom legend handles in EXACT desired order
    from matplotlib.lines import Line2D
    legend_labels = ['Geo', 'Enriched', 'Topo', 'Geo + Topo']
    handles = [
        Line2D([0], [0], color='C0', lw=4, label=legend_labels[0]),
        Line2D([0], [0], color='C1', lw=4, label=legend_labels[1]),
        Line2D([0], [0], color='C2', lw=4, label=legend_labels[2]),
        Line2D([0], [0], color='C3', lw=4, label=legend_labels[3])
    ]
    
    # Position legend BELOW plot in 2x2 grid
    ax.legend(
        handles=handles,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.15),  # 15% below plot center
        ncol=2,                        # 2 columns
        frameon=False,                 # No border
        fontsize=11*2.5
    )
    
    plt.xlabel("PLD Decile", )
    plt.ylabel("Mean Absolute Error", )
    plt.tight_layout(rect=[0, 0, 1, 0.93])  # Reserve 5% space at bottom for legend
    plt.savefig("supplementary_assets/SI_figure4_pld_regime.pdf", bbox_inches='tight')
    plt.close()
    
    # -- SI Figure 5: Residual Analysis (Best Geometry-Only Model) --
    geom_hgb = hgb_data[hgb_data['family'] == 'geometry_only'].copy()
    geom_hgb['residual'] = geom_hgb['y_true'] - geom_hgb['y_pred']
    
    # -- SI Figure 5: Residual Plot (Horizontal Layout) --
    plt.figure(figsize=(12, 6))  # Wider than tall for horizontal layout
    sns.scatterplot(
        data=geom_hgb, 
        x='y_pred', 
        y='residual', 
        alpha=0.1,
        edgecolor='none'
    )
    plt.axhline(0, color='red', linestyle='--', linewidth=1.5*2.5)
    plt.xlabel("Predicted Uptake (mmol/g)", )
    plt.ylabel("Residual (mmol/g)",)
    plt.tight_layout()
    plt.savefig("supplementary_assets/SI_figure5_residuals.pdf", bbox_inches='tight')
    plt.close()
    
    # -- SI Figure 6: Topology Errors (Horizontal Barplot) --
    # Sort DataFrame by 'abs_error' descending BEFORE plotting
    topology_avg = geom_hgb.groupby('topology_label')['abs_error'].agg(['mean', 'std']).reset_index()
    topology_avg.columns = ['topology_label', 'mean_abs_error', 'std_error']
    
    # Step 2: Sort by mean error (descending)
    sorted_topology = topology_avg.sort_values('mean_abs_error', ascending=False)
    
    # Step 3: Create horizontal barplot from sorted data
    plt.figure(figsize=(10, 12))
    ax = sns.barplot(
        data=sorted_topology,
        x='mean_abs_error',
        y='topology_label',
        orient='h',
        errorbar=('ci', 95),  # 95% confidence interval (standard for journals)
        capsize=0.1,          # Error bar cap size
        errcolor='black',     # Error bar color
        errwidth=1.5          # Error bar thickness
    )
    
    # Add value labels to bars

    
    plt.xlabel("Mean Absolute Error (mmol/g)", labelpad=20)
    plt.ylabel("Topology", labelpad=30)
    plt.tight_layout()
    plt.savefig("supplementary_assets/SI_figure6_topology_errors.pdf", bbox_inches='tight')
    plt.close()
# # ==========================================
# # 2. REGIME-RESOLVED TABLES (TABLE 7)
# # ==========================================
def generate_regime_tables(df, preds):
    print("Generating Regime-Resolved Tables (Table 7 equivalents)...")


# Apply to both DataFrames before merging
    df = optimize_data_types(df)
    preds = optimize_data_types(preds)
    merged = preds.merge(df, on=ID_COL, how="left")
    merged['abs_error'] = (merged['y_true'] - merged['y_pred']).abs()
    
    # Calculate deciles
    merged['density_decile'] = pd.qcut(merged['Density'], 10, labels=False, duplicates='drop')
    merged['pld_decile'] = pd.qcut(merged['Df'], 10, labels=False, duplicates='drop')
    merged['uptake_decile'] = pd.qcut(merged['y_true'], 10, labels=False, duplicates='drop')

    # We use HistGradientBoosting as the representative model for the tables
    hgb_data = merged[merged['model'] == 'hgb']

    for regime in ['density_decile', 'pld_decile', 'uptake_decile']:
        # Pivot table to show MAE for each family across the bins
        table = hgb_data.pivot_table(
            index=regime, 
            columns='family', 
            values='abs_error', 
            aggfunc=['count', 'mean']
        )
        
        # Clean up columns for export
        table.columns = ['_'.join(col).strip() for col in table.columns.values]
        # Keep only one count column (since all families evaluate the same structures)
        count_col = [c for c in table.columns if 'count' in c][0]
        mae_cols = [c for c in table.columns if 'mean' in c]
        
        final_table = table[[count_col] + mae_cols].rename(columns={count_col: "Structure_Count"})
        final_table.to_csv(f"results/tables/Table7_{regime}_Performance.csv")

# ==========================================
# 3. ROBUSTNESS & SENSITIVITY CHECKS
# ==========================================
def run_sensitivity_checks(df):
    print("Running Fast Robustness & Sensitivity Checks...")
    
    # Use just Seed 0 from the splits to save computation time for the SI checks
    with open("data_processed/split_definitions/random_split_seed_0.json", "r") as f:
        split_data = json.load(f)
        
    train_df = df[df[ID_COL].isin(set(split_data["train_ids"]))]
    test_df = df[df[ID_COL].isin(set(split_data["test_ids"]))]
    
    y_train = train_df[TARGET_COL]
    y_test = test_df[TARGET_COL]
    
    results = []

    # Setup a standard fast model pipeline (HistGradientBoosting requires no scaling/imputation for basic numeric data)
    model = HistGradientBoostingRegressor(random_state=42)

    # Check 1: Compact 5-Feature Geometry vs Enriched
    compact_5 = ['Density', 'ASA', 'Df', 'Di', 'AVAf']
    
    model.fit(train_df[compact_5], y_train)
    r2_compact = r2_score(y_test, model.predict(test_df[compact_5]))
    results.append({"Test": "Compact 5-Feature Ablation", "Metric": "R2", "Value": r2_compact})

    model.fit(train_df[ENRICHED_COLS], y_train)
    r2_enriched = r2_score(y_test, model.predict(test_df[ENRICHED_COLS]))
    results.append({"Test": "Full Enriched Feature Set", "Metric": "R2", "Value": r2_enriched})

    # Check 2: Log-Transformed Target Scaling
    y_train_log = np.log1p(y_train)
    y_test_log = np.log1p(y_test)
    
    model.fit(train_df[GEOMETRY_COLS], y_train_log)
    preds_log = model.predict(test_df[GEOMETRY_COLS])
    preds_back_transformed = np.expm1(preds_log) # Convert back to native scale for fair MAE comparison
    
    mae_log_pipeline = mean_absolute_error(y_test, preds_back_transformed)
    results.append({"Test": "Target Log Transformation (Back-transformed MAE)", "Metric": "MAE", "Value": mae_log_pipeline})

    model.fit(train_df[GEOMETRY_COLS], y_train)
    mae_native = mean_absolute_error(y_test, model.predict(test_df[GEOMETRY_COLS]))
    results.append({"Test": "Native Target Scale", "Metric": "MAE", "Value": mae_native})

    # Save sensitivity results
    pd.DataFrame(results).to_csv("results/tables/robustness/Sensitivity_Checks_Summary.csv", index=False)

    # Check 3: Permutation Feature Importance for the Enriched Set
    print("  Calculating Permutation Importance (Random Forest)...")
    rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
    
    # Impute missing values just for RF
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(train_df[ENRICHED_COLS])
    X_test_imp = imputer.transform(test_df[ENRICHED_COLS])
    
    rf.fit(X_train_imp, y_train)
    from joblib import parallel_backend
    with parallel_backend('threading'):
        r = permutation_importance(rf, X_test_imp, y_test, n_repeats=5, random_state=42, n_jobs=-1)
    
    importance_df = pd.DataFrame({
        "Feature": ENRICHED_COLS,
        "Importance_Mean": r.importances_mean,
        "Importance_Std": r.importances_std
    }).sort_values(by="Importance_Mean", ascending=False)
    
    importance_df.to_csv("results/tables/robustness/Permutation_Feature_Importance.csv", index=False)
    
    # Plot Feature Importance
    plt.figure(figsize=(10, 8))
    ax=sns.barplot(data=importance_df.head(10), x='Importance_Mean', y='Feature', color='steelblue')
    plt.xlabel("Mean Importance")
    ax.set_title("")
    plt.tight_layout()
    plt.savefig("supplementary_assets/SI_figure_feature_importance.pdf")
    plt.close()

# # ==========================================
# # EXECUTION
# # ==========================================
if __name__ == "__main__":
    try:
        preds_df = pd.read_csv(PREDS_FILE)
        metrics_df = pd.read_csv(RAW_METRICS_FILE)
    except FileNotFoundError:
        print(f"ERROR: Cannot find {PREDS_FILE} or {RAW_METRICS_FILE}.")
        print("Please ensure you have completely run the Main Manuscript script first.")
        exit(1)

    df_clean = load_and_prep_data()
    
    dff= generate_si_figures(df_clean, preds_df, metrics_df)
    generate_regime_tables(df_clean, preds_df)
    run_sensitivity_checks(df_clean)
    
    print("\nSI Analysis Complete! Check the 'supplementary_assets' and 'results/tables' folders.")

Generating SI Figures 1-6...
Generating Regime-Resolved Tables (Table 7 equivalents)...
Running Fast Robustness & Sensitivity Checks...
  Calculating Permutation Importance (Random Forest)...

SI Analysis Complete! Check the 'supplementary_assets' and 'results/tables' folders.
